# `tensorium` tutorial, part 2: metric geometry


In [1]:
from sympy import simplify, symbols
from tensorium import *

S2 = Manifold("S^2", 2)
U_N = OpenSet("U_N", S2)  # S^2 without the north pole
U_S = OpenSet("U_S", S2)  # S^2 without the south pole
x, y = symbols("x y", real=True)
u, v = symbols("u v", real=True)
X_N = Chart("X_N", U_N, (x, y))
X_S = Chart("X_S", U_S, (u, v), relations={X_N: (u/(u**2 + v**2), v/(u**2 + v**2))})
atlas = Atlas(S2, [X_N, X_S])
S2.set_atlas(atlas)
xN, yN = X_N.symbols
uS, vS = X_S.symbols
f_N_local = LocalTensorField(X_N, (0, 0), 1/(1 + xN**2 + yN**2))
f_S_local = LocalTensorField(X_S, (0, 0), (uS**2 + vS**2)/(1 + uS**2 + vS**2))
f = TensorField(S2, (0, 0), {X_N: f_N_local, X_S: f_S_local}, ())
V_N_local = LocalTensorField(X_N, (1, 0), [-yN, xN])
V_S_local = LocalTensorField(X_S, (1, 0), [-vS, uS])
V = Vector(S2, {X_N: V_N_local, X_S: V_S_local})
omega_N_local = LocalTensorField(X_N, (0, 1), [-2*xN/(1 + xN**2 + yN**2)**2, -2*yN/(1 + xN**2 + yN**2)**2])
omega_S_local = LocalTensorField(X_S, (0, 1), [2*uS/(1 + uS**2 + vS**2)**2, 2*vS/(1 + uS**2 + vS**2)**2])
omega = OneForm(S2, {X_N: omega_N_local, X_S: omega_S_local})
W_N_local = LocalTensorField(X_N, (1, 0), [xN, yN])
W_S_local = LocalTensorField(X_S, (1, 0), [-uS, -vS])
W = Vector(S2, {X_N: W_N_local, X_S: W_S_local})
T = V * omega


## 2. Metric geometry


The previous section only used the differentiable structure of the manifold. We now add a metric and turn the same abstract sphere into a `MetricManifold`.

We use the round metric on $S^2$ in stereographic coordinates:

$$
g = \frac{4}{(1+x^2+y^2)^2}(dx\otimes dx+dy\otimes dy),
$$

with the analogous expression in the south chart. Giving both local expressions keeps the metric defined on the whole sphere.

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
In the library, when working with metric geometry, it is usually cleaner to first define the metric as an ordinary tensor field on the underlying manifold, and only then build the metric manifold as the pair (M,g). Once this pair has been created, the remaining tensor fields and geometric objects should be defined directly over the metric manifold.
</div>


In [2]:
lambda_N = 4/(1 + xN**2 + yN**2)**2
lambda_S = 4/(1 + uS**2 + vS**2)**2

g_N_local = LocalCovariantMetricTensor(X_N, (lambda_N, 0, lambda_N))
g_S_local = LocalCovariantMetricTensor(X_S, (lambda_S, 0, lambda_S))

g = CovariantMetricTensor(S2, {X_N: g_N_local, X_S: g_S_local})
g_inv = g.inverse()
S2_metric = MetricManifold(S2, covariant_metric=g, contravariant_metric=g_inv)

Display(S2_metric)
Display(g, name="g")
Display(g_inv, name=r"g^{-1}")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Metrics are implemented through specific local and global classes that inherit from the general tensor-field infrastructure. Since a metric is symmetric, these classes already include the corresponding symmetry information. Therefore, when defining a metric locally, it is enough to provide the diagonal entries and the components above the diagonal; the remaining components are recovered automatically by symmetry.
</div>

### 2.1. Raising and lowering indices


Once the metric and its inverse are known, tensor indices can be raised or lowered. For instance, the rotation vector field $V$ can be converted into the corresponding one-form $g(V,\cdot)$.

The original fields were defined on the bare manifold `S2`. To use metric operations, we view the same local data as fields on `S2_metric`.


In [3]:
V_metric = Vector(S2_metric, V.local_representations)
omega_metric = OneForm(S2_metric, omega.local_representations)

V_flat = V_metric.raise_lower_indices((0,))
omega_sharp = omega_metric.raise_lower_indices((0,))

Display(V_flat, name=r"g(V,\cdot)")
Display(omega_sharp, name=r"g^{-1}(\omega,\cdot)")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 2.2. Levi-Civita connection


The Levi-Civita connection is the canonical affine connection associated with a metric: it is torsion-free and metric-compatible. In local coordinates it is represented by the Christoffel symbols $\Gamma^\rho{}_{\mu\nu}$, computed from the metric and its first derivatives.

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
At the implementation level, `LeviCivitaConnection` is a specialized case of the more primitive affine-connection infrastructure. The library currently provides an automatic constructor for Levi-Civita connections, because this is the case most commonly needed in Riemannian and pseudo-Riemannian geometry. However, the lower-level connection classes can also be used directly to define other affine connections by explicitly providing their local coefficients.
</div>


In [4]:
Gamma = LeviCivitaConnection(S2_metric)

Display(Gamma, name=r"\nabla")
Display(Gamma.local_representation(X_N), name=r"\Gamma")

<IPython.core.display.Math object>

'Known local representations: X_N, X_S'

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 2.3. Affine covariant derivatives


In this section we only use the affine connection induced by the metric. Later, when internal indices are introduced, the same general idea will be extended with gauge contributions.

A `CovariantDerivative` is an operator. In the purely affine case, applying it to a tensor field adds one covariant geometric index:

$$
\nabla: \mathcal{T}^{r}_{s}(M) \longrightarrow \mathcal{T}^{r}_{s+1}(M).
$$

For scalars it reduces to the ordinary differential, while for vector and tensor fields it also includes the Christoffel-symbol correction terms.


In [5]:
nabla = CovariantDerivative(affine_connection=Gamma)

f_metric = TensorField(S2_metric, (0, 0), f.local_representations, ())
T_metric = V_metric * omega_metric

nabla_f = nabla(f_metric)
nabla_V = nabla(V_metric)
nabla_T = nabla(T_metric)

Display(nabla_f, chart=X_N, name=r"\nabla f")
Display(nabla_V, chart=X_N, name=r"\nabla V")
Display(nabla_T, chart=X_N, name=r"\nabla T")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">

`CovariantDerivative` inherits from the more primitive `TensorOperator` class. This means that $\nabla$ can be stored and reused as an operator acting on arbitrary tensor fields of type $(r,s)$, instead of being only a one-shot function. This design will become useful in later notebooks, where tensor operators can be combined, contracted, or used as building blocks for more elaborate constructions.

If we only want the result directly, without explicitly storing the operator, the convenience function is `covariant_derivative(field, affine_connection=Gamma)`.
</div>

### 2.4. Curvature


The Riemann tensor is computed from the affine connection. For the round sphere it is non-zero, because $S^2$ is curved. The Ricci tensor satisfies
$
\operatorname{Ric}=g
$
for the unit two-sphere.


In [6]:
Riemann = riemann_from_affine_connection(Gamma)
Ricci = Riemann.contraction(0, 2)
g_on_metric = TensorField(S2_metric, (0, 2), g.local_representations, (-1, -1))
Ricci_minus_g = Ricci - g_on_metric

Display(Riemann, name="R")
Display(Ricci, name=r"\mathrm{Ric}")
Display(Ricci_minus_g, name=r"\mathrm{Ric}-g")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>